# Downscale using the QPLAD method

The transfer function from coarse to high-res is entirely done on the reference (reanalysis) product, between its 1-deg regridded and 0.25-deg regridded forms. 

$$ af(q) = B^{-1}_{o,h,f}(q) - F^{-1}_{o,h,c}(q)$$

where $o, h, c = $ obs (reanalysis), historical, coarse and $f = $ fine.

Workflow: 

First,
1. Regrid reanalysis to 0.25 grid, save
2. Tile 1-deg reanalysis (previously regridded) to 0.25 grid using nearest neighbors, save

Then,
1. Load model data
2. Reshape model data for rolling
3. Get pctof each model data value
4. Get change in that quantile between coarse and fine for reanalysis
5. Add that change to model data
6. Use now downscaled model data to calculate desired data for damage function
7. Save what the resultant`lat x lon x parameter` array

In [24]:
import xarray as xr
import numpy as np
import pandas as pd
import fsspec
from tqdm.notebook import tqdm
import re
import os
from numba import jit as njit
import zarr
import dask
from distributed import Client
import warnings

from funcs_support import get_filepaths, get_params, utility_save
from funcs_preprocessing import preprocess_models
from funcs_processing import (sliding_ranks, calc_quantiles,
                              dmgf_params_bins, dmgf_params_carleton, calc_equantile_diffs,
                              calc_pctof_bcmodels)
from funcs_aux import (_create_filenames, _restore_doys, dask_isel, _load_gwls,
                       get_landmask, repeat_ds, _verify_gwl_range)

dir_list = get_params()

In [25]:
# Start dask client
client = Client()
display(client)

/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 36505 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc3/proxy/36505/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc3/proxy/36505/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:33263,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc3/proxy/36505/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:43193,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/bcd_me_proc3/proxy/41389/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:39845,


2026-02-15 13:44:40,435 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 018dad4082bf49b41a1c9c16ddea0753 initialized by task ('rechunk-merge-rechunk-transfer-e5e403fd65d587080473a0c8daabfe65', 4, 9, 0, 0, 4, 9, 0, 0) executed on worker tcp://127.0.0.1:39973
2026-02-15 13:44:47,319 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f814740761092d20c0c26f67499b9484 initialized by task ('rechunk-merge-rechunk-transfer-e5e403fd65d587080473a0c8daabfe65', 4, 8, 0, 0, 4, 8, 0, 0) executed on worker tcp://127.0.0.1:39973
2026-02-15 13:44:48,192 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 018dad4082bf49b41a1c9c16ddea0753 deactivated due to stimulus 'task-finished-1771188288.1912358'
2026-02-15 13:44:49,140 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 82aefe1c86727dd962c9b09fdfeb1d91 initialized by task ('rechunk-merge-rechunk-transfer-e5e403fd65d587080473a0c8daabfe65', 4, 6, 0, 0, 4, 6, 0, 0) executed on worker tcp://127.0.0.1:33653
2026-02-15

In [12]:
#----------- Setup -----------
params_var = {'var':'tas','freq':'day'}
#params_var = {'var':'tasmax','freq':'day'}

params_subset = {'lat':slice(-56,86),'lon':slice(-180,180)}
params_proc = {'wwidth':31,'nqs':100,'bc_type':'qdm'}
params_param = ['sumpoly','binC','binF']

gwl_info_rea = pd.Series({'warming_level':0.61,'start_year':1982,'end_year':2001})
# Get reference 0.25x0.25 grid that slots into the 
# ref_grid 1x1 grid perfectly 
fine_grid = xr.Dataset(coords = {'lat':(['lat'],np.arange(params_subset['lat'].start+0.125,
														 params_subset['lat'].stop-0.124,0.25)),
								'lon':(['lon'],np.arange(params_subset['lon'].start+0.125,
														 params_subset['lon'].stop-0.124,0.25))})

# Get filesystem
fs = fsspec.implementations.local.LocalFileSystem()

# Get model filepath info
df = get_filepaths()
df = df.query('varname == "'+params_var['var']+'" and '+
              'freq == "'+params_var['freq']+'"')
# Get global files (no suffix)
if ('suffix' not in params_var) and 'suffix' in df.columns:
    df = df.where(df.suffix != df.suffix).dropna(how='all')

fns_out = _create_filenames(gwl_info_rea,params_var,params_proc)

## Create coarse-to-fine downscaling map from reanalysis

In [8]:
# 2 hours on 18 CPUs for the four base reanalyses. GMFD took longer because of chunking, I believe. 
for mod_rea in ['ERA5-025','GMFD','MERRA2','JRA-3Q']:
    params_proc['mod_rea'] = mod_rea

    fns_out = _create_filenames(gwl_info_rea,params_var,params_proc)
    
    #----------- Create coarse-to-fine reanalysis map -----------
    # Get change in each quantile between coarse and fine for reanalysis
    if not fs.exists(fns_out['rea_rgrd_diffs']):
    
        #----------- Regrid reanalysis to fine -----------
        ## Preprocess reanalysis by regridding to 0.25
        # (Regrid, rechunk, reshape)
        preprocess_models(file_rows=df.query('model == "'+params_proc['mod_rea']+'"'),
                          gwl_info=gwl_info_rea,
                          params_var=params_var,
                          params_proc=params_proc,
                          params_subset=params_subset,
                          ref_grid=fine_grid,
                          extra_filename_slots=['FINEgrid'],
                          regrid_method='bilinear',
                          chunk_sizes = {'geo':20,'time':100}, # SMALLEST FOR PRISM # smaller chunk size for fine regridding
                          fs=fs)

        if mod_rea == 'PRISM':
            # HACK BECAUSE PRISM HAS "OBS" AS A RUN ID
            os.system('mv  /glade/derecho/scratch/schwarzwald/climate_proc/PRISM/tasmax_day_PRISM_historical_obs_FINEgrid_CACHE-RESHAPE_CONUS.zarr '+
                      '/glade/derecho/scratch/schwarzwald/climate_proc/PRISM/tasmax_day_PRISM_historical_reanalysis_FINEgrid_CACHE-RESHAPE_CONUS.zarr')
        
        #----------- Tile coarse reanalysis to fine -----------
        if not fs.exists(fns_out['reshp_rea_finefromcoarse']):
            # Load 1deg reanalysis
            ds = xr.open_zarr(fns_out['reshp_rea'])
        
            # Repeat (copy each element) by a factor of 4 in lat/lon
            # directions to match downscaled scale (this is equivalent
            # to nearest-neighbor matching)
            ds = repeat_ds(ds,4)
            
            # For some reason, this messes up chunking sometimes
            ds = ds.chunk({'lat':20,'lon':20})
            
            # Export
            utility_save(ds.drop_encoding(),fns_out['reshp_rea_finefromcoarse'])
        else:
            print(fns_out['reshp_rea_finefromcoarse']+' exists, skipped!')
    
        # Load fine and coarse-to-fine reanalysis
        ds_rea = xr.merge([xr.open_zarr(fns_out['reshp_rea_fine']).rename({params_var['var']:params_var['var']+'fine'}),
                           xr.open_zarr(fns_out['reshp_rea_finefromcoarse']).rename({params_var['var']:params_var['var']+'NN'})])
        
        # Make year general to fit with futre model data
        ds_rea['year'] = np.arange(1,ds_rea.sizes['year']+1,dtype=int)
    
        # Get 1deg landmask
        landmask = get_landmask(ds_rea)
        
        # Restrict to just land pixels
        ds_rea = ds_rea.where(landmask)
    
        # Calculate change in each quantile between coarse and fine grids
        qdiff = xr.apply_ufunc(calc_equantile_diffs,
                               ds_rea[params_var['var']+'NN'].astype(np.float32),
                               ds_rea[params_var['var']+'fine'].astype(np.float32),
                               input_core_dims = [['dayofyear','year'],
                                                  ['dayofyear','year']],
                               output_core_dims = [['dayofyear','quantile']],
                               kwargs = {'wwidth':params_proc['wwidth']},
                               dask = 'parallelized',
                               dask_gufunc_kwargs={'output_sizes':{'quantile':params_proc['wwidth']*ds_rea.sizes['year']}},
                               output_dtypes=[np.float32],
                               vectorize=True).to_dataset(name=params_var['var'])

        # Sometimes this messes up the chunks... 
        qdiff = qdiff.chunk({'lat':20,'lon':20})

        qdiff.attrs = {'SOURCE':'downscale_qplad.ipynb',
                              'DESCRIPTION':'Downscaling map between 1 and 0.25 degs; differences of rolling ecdfs.',
                              'LANDMASK_FLAG':'True',
                              'wwidth':params_proc['wwidth']}
        
        # Save map from coarse to fine in ERA5
        utility_save(_restore_doys(qdiff.drop_encoding(),params_proc),fns_out['rea_rgrd_diffs'],keep_chunk_encoding=True,
                     )
    else:
        print(fns_out['rea_rgrd_diffs']+' exists, skipped!')

    # Remove temporary files
    for fn in ['reshp_rea_fine','reshp_rea_finefromcoarse']:
        if fs.exists(fns_out[fn]):
            fs.rm(fns_out[fn],recursive=True)
            print('Intermediary file '+fns_out[fn]+' removed!')
               

/glade/derecho/scratch/schwarzwald/bcd_me/ERA5-025/tasqdiff_day_ERA5-025_historical_reanalysis_19820101-20011231_GWL0-61_COARSE-FINE-scale.zarr exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/GMFD/tasqdiff_day_GMFD_historical_reanalysis_19820101-20011231_GWL0-61_COARSE-FINE-scale.zarr exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/MERRA2/tasqdiff_day_MERRA2_historical_reanalysis_19820101-20011231_GWL0-61_COARSE-FINE-scale.zarr exists, skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/JRA-3Q/tasqdiff_day_JRA-3Q_historical_reanalysis_19820101-20011231_GWL0-61_COARSE-FINE-scale.zarr exists, skipped!


## Phase 2 (by-model-run processing)
1. Load (bias-corrected) model data
2. Reshape model data for rolling
3. Get pctof each model data value
5. Add reanalysis coarse-to-fine map to model data
6. Use now downscaled model data to calculate desired data for damage function
7. Save what should now be a `lat x lon x parameter` array
8. Remove temporary files

In [26]:
#----------- Processing Models -----------
# Get GWL info
gwl_info = _load_gwls()

# Get model filepath info
df = get_filepaths()
df = df.query('varname == "'+params_var['var']+'" and '+
              'freq == "'+params_var['freq']+'"')
# Get global files (no suffix)
if ('suffix' not in params_var) and 'suffix' in df.columns:
    df = df.where(df.suffix != df.suffix).dropna(how='all')

keys_all = [k for k in gwl_info.groupby(['model','ensemble','exp']).groups]
keys_all = [k for k in keys_all if k in df.groupby(['model','run','exp']).groups]

# Subset to just files that have both historical and future data 
# (saved under the future experiment name, not 'historical') 
keys_all = [k for k in keys_all if k[2] != 'historical']


In [27]:
keep_dwscld_tseries = True

for keys in tqdm(keys_all):
    print('\n\n***********************\nProcessing '+', '.join(keys)+'!\n***********************')
    
    # GWL parameters for this model / exp / run
    gwl_info_tmp = gwl_info.reset_index().set_index(['model','ensemble','exp']).sort_index().loc[keys,:]
    # Local filepath parameters for this model / exp / run
    df_tmp = df.query('model == "'+keys[0]+'" and run == "'+keys[1]+'" and ('+
                      'exp == "historical" or exp == "'+keys[2]+'")')
    
    # Double-check that local files are present for all needed
    try:
        gwl_info_tmp = _verify_gwl_range(df_tmp,gwl_info_tmp,gwl_info_rea)
    except Exception as e:
        warnings.warn(str(e)+' Skipping.')
        continue

    for mod_rea in ['ERA5-025','GMFD','MERRA2','JRA-3Q']: #,'JRA-3Q']:['PRISM']
        params_proc['mod_rea'] = mod_rea

        # Create filenames for 
        fns_out = _create_filenames(gwl_info_rea,params_var,params_proc,file_rows=df_tmp)

        if keep_dwscld_tseries:
            fns_out['dwscld_'+params_proc['bc_type']] = re.sub(dir_list['proc'],
                   dir_list['proc_fullds'],
                   fns_out['dwscld_'+params_proc['bc_type']])

        # Create placeholder filename - empty file created to show that this instance
        # is working on this set of files. nc so it shows up in get_filepaths(). Will 
        # be deleted once the processing is complete. 
        placeholder_fn = (dir_list['proc']+keys[0]+'/'+'_'.join([params_var['var']+'tmpproc',params_var['freq'],keys[0],keys[2],keys[1],
                                                     params_proc['bc_type']+'-'+'qplad',mod_rea])+'_TMP.nc')
        
        if np.any([not os.path.exists(fns_out[v+'_'+params_proc['bc_type']]+'/.done') for v in params_param]):
            if fs.exists(placeholder_fn):
                print('Another instance is working on this downscaling run, skipping for now.')
                continue
            
            if not fs.exists(fns_out['biascrct_'+params_proc['bc_type']]+'/.done'):
                print('Bias-corrected file '+fns_out['biascrct_'+params_proc['bc_type']]+' does not exist (or is not finished processing), skipping.')
                continue
                
            #----------- Create empty placeholder file -----------
            open(placeholder_fn, 'w').close()
            
            #----------- Model quantile calculations -----------
            if not fs.exists(fns_out['pctof_tiled']+'/.done'):
                # Calculate rolling percentile of each value in models
                # These are almost, but not quite identical (corrs 0.99X, 
                # departures from the 1:1 seem random), between reanalyses, 
                # possibly because of some complex interaction between the 
                # QDM procedure and calculating the rolling percentile here
                # so have to run it for every reanalysis separately
                calc_pctof_bcmodels(file_rows=df_tmp,
                                      gwl_info_rea=gwl_info_rea,
                                      params_var=params_var,
                                      params_proc=params_proc,
                                      fs=fs)
            
            # Load rolling percentiles
            ds_pctof = xr.open_zarr(fns_out['pctof_tiled']) 

            #----------- Model downscaling -----------
            ## Repeat element-wise model data to fine resolution
            if not os.path.exists(fns_out['dwscld_'+params_proc['bc_type']]+'/.done'):
                # Open bias-corrected data
                ds = xr.open_zarr(fns_out['biascrct_'+params_proc['bc_type']])
                # Tile to 0.25 grid
                ds = repeat_ds(ds, 4).chunk({'lat':20,'lon':20})
            
                # Turn pcts_of into integer indices (in python 0-based indexing format)
                idxs = (ds_pctof[params_var['var']]*
                        params_proc['wwidth']*ds.sizes['year']).astype(int) - 1
                
                ## Add qdiff change to model data
                # Load reanalysis quantile difference from coarse to fine grids
                qdiff = xr.open_zarr(fns_out['rea_rgrd_diffs'])
                # Just to preclude lingering xesmf issues... 
                qdiff = qdiff.sel(lat=ds.lat, lon=ds.lon).chunk({'lat':20,'lon':20})
                
                # Downscale by adding each quantile's coarse-to-fine map from reanalysis
                ds_dscld = ds + dask_isel(qdiff[params_var['var']],'quantile',idxs)
    
                # Save temporary file (yes, this speeds things up like crazy) 
                ds_dscld.attrs = {'SOURCE':'downscale_qplad.ipynb',
                      'DESCRIPTION':'QPLAD downscaling of QDM bias-corrected data by GWL, using '+params_proc['mod_rea']+' as a reference base.',
                      'REF_DATA':params_proc['mod_rea']}
                utility_save(ds_dscld.drop_encoding(),fns_out['dwscld_'+params_proc['bc_type']],save_kwargs = {'zarr_format':3},
                             zarr_mode = 'w')

            # Load temporary file 
            ds_dscld = xr.open_zarr(fns_out['dwscld_'+params_proc['bc_type']])
            
            #----------- Parameter calculation -----------
            if ('sumpoly' in params_param) and (not fs.exists(fns_out['sumpoly_'+params_proc['bc_type']]+'/.done')):
                # Calculate sums of polynomials, used e.g. by the Carleton et al.
                # damage function, make sure it's in C if a temperature variable
                ds_tmp = (dmgf_params_carleton((ds_dscld[params_var['var']]-
                                                    ('tas' in params_var['var'])*273.15), 
                                                   np.arange(1,5)).
                              to_dataset(name=params_var['var']+'sumpoly'))
                ds_tmp = ds_tmp.mean('year')
                
                ds_tmp[params_var['var']+'sumpoly'].attrs['long_name'] = 'average summed polynomials of daily '+params_var['var']+' per year'
                ds_tmp[params_var['var']+'sumpoly'].attrs['units'] = 'sum of C^k'
                
                ds_tmp = ds_tmp.rename({'param':'degree'})
                ds_tmp['degree'].attrs['long_name'] = 'polynomial degree'
                ds_tmp['degree'] = [int(p[1:]) for p in ds_tmp.degree.values]
                
                # General attributes
                ds_tmp.attrs['SOURCE'] = 'downscale_qplad.ipynb'
                ds_tmp.attrs['DESCRIPTION'] = 'average summed polynomials of '+params_var['var']+' per year, calculated over 20-year periods centered around the year in which a given (smoothed using a 20-year-wide gaussian kernel) GWL is first crossed in GMST'
            
                # Chunk to ~O(1MB) size chunks
                ds_tmp = ds_tmp.chunk({'lat':100,'lon':100,'gwl':-1,'degree':-1})
                
                utility_save(ds_tmp.drop_encoding(),fns_out['sumpoly_'+params_proc['bc_type']],save_kwargs={'zarr_format':3},
                             zarr_mode = 'w')
            
            if ('binC' in params_param) and (not fs.exists(fns_out['binC_'+params_proc['bc_type']]+'/.done')):
                # Define 1 deg C bins
                binsC = np.arange(-21,51) + 0.5
                # Calculate days in bins (converting to C from K)
                ds_tmp = (dmgf_params_bins(ds_dscld[params_var['var']]-273.15,bins=binsC).
                                     to_dataset(name=params_var['var']+'binC'))
                ds_tmp = ds_tmp.mean('year')
            
                # Add bin bounds, for reference
                ds_tmp['bin_bnds'] = xr.DataArray(np.reshape(np.array([-np.inf,*np.repeat(binsC,2),np.inf]),(len(binsC)+1,2)),
                                                        dims=('bin', 'bnds'),
                                                        coords={'bin':ds_tmp.bin,
                                                                  'bnds':(('bnds'),[0,1])})
            
                ds_tmp[params_var['var']+'binC'].attrs['long_name'] = 'days in '+params_var['var']+' bins'
                ds_tmp[params_var['var']+'binC'].attrs['units'] = 'days / year'
            
                ds_tmp['bin'].attrs['units'] = 'deg C'
                ds_tmp['bin'].attrs['long_name'] = params_var['var']+' temperature bin center'
            
                ds_tmp['bin_bnds'].attrs['units'] = 'deg C'
                ds_tmp['bin_bnds'].attrs['long_name'] = params_var['var']+' temperature bin bounds'
            
                # General attributes
                ds_tmp.attrs['SOURCE'] = 'downscale_qplad.ipynb'
                ds_tmp.attrs['DESCRIPTION'] = 'average days in '+params_var['var']+' bins per year, calculated over 20-year periods centered around the year in which a given (smoothed using a 20-year-wide gaussian kernel) GWL is first crossed in GMST'
            
                # Chunk to ~O(1MB) size chunks
                ds_tmp = ds_tmp.chunk({'lat':100,'lon':100,'gwl':1,'bin':-1})
                
                utility_save(ds_tmp.drop_encoding(),fns_out['binC_'+params_proc['bc_type']],save_kwargs={'zarr_format':3},
                             zarr_mode = 'w')
            
            if ('binF' in params_param) and (not fs.exists(fns_out['binF_'+params_proc['bc_type']]+'/.done')):
                # Define 5 deg F bins (as floats, since within the numba function
                # infinities are added to the sides, which changes the type to float
                # if integer. Numba doesn't allow dtype changes within the njit'ed 
                # function, this throws an error). 
                #binsF = np.arange(0, 101, 5) - 32
                binsF = np.arange(0, 130, 5).astype(float) 
                # Calculate days in bins (converting to F from K)
                ds_tmp = (dmgf_params_bins(ds_dscld[params_var['var']]*(9/5) - 459.67, bins=binsF).
                            to_dataset(name=params_var['var']+'binF'))
                ds_tmp = ds_tmp.mean('year')
            
                # Add bin bounds, for reference
                ds_tmp['bin_bnds'] = xr.DataArray(np.reshape(np.array([-np.inf,*np.repeat(binsF,2),np.inf]),(len(binsF)+1,2)),
                                                        dims=('bin', 'bnds'),
                                                        coords={'bin':ds_tmp.bin,
                                                                  'bnds':(('bnds'),[0,1])})
            
                ds_tmp[params_var['var']+'binF'].attrs['long_name'] = 'days in '+params_var['var']+' bins'
                ds_tmp[params_var['var']+'binF'].attrs['units'] = 'days / year'
            
                ds_tmp['bin'].attrs['units'] = 'deg F'
                ds_tmp['bin'].attrs['long_name'] = params_var['var']+' temperature bin center'
            
                ds_tmp['bin_bnds'].attrs['units'] = 'deg F'
                ds_tmp['bin_bnds'].attrs['long_name'] = params_var['var']+' temperature bin bounds'
            
                # General attributes
                ds_tmp.attrs['SOURCE'] = 'downscale_qplad.ipynb'
                ds_tmp.attrs['DESCRIPTION'] = 'average days in '+params_var['var']+' bins per year, calculated over 20-year periods centered around the year in which a given (smoothed using a 20-year-wide gaussian kernel) GWL is first crossed in GMST'
            
                # Chunk to ~O(1MB) size chunks
                ds_tmp = ds_tmp.chunk({'lat':100,'lon':100,'gwl':1,'bin':-1})
                
                utility_save(ds_tmp.drop_encoding(),fns_out['binF_'+params_proc['bc_type']],save_kwargs={'zarr_format':3},
                             zarr_mode = 'w')
                
            
            if ('mx5d' in params_param) and (not fs.exists(fns_out['mx5d_'+params_proc['bc_type']]+'/.done')):
                # Calculate mean across years of 5 hottest days per year
                ds_tmp = dmgf_params_max5d(ds_dscld[params_var['var']],5).to_dataset(name=params_var['var']+'mx5d')
            
                ds_tmp[params_var['var']+'mx5d'].attrs['long_name'] = 'Mean annual 5-day max '+params_var['var']
                ds_tmp[params_var['var']+'mx5d'].attrs['units'] = 'K'
            
                # General attributes
                ds_tmp.attrs['SOURCE'] = 'downscale_qplad.ipynb'
                ds_tmp.attrs['DESCRIPTION'] = ("Interannual mean of each year's maximum 5-day "+params_var['var']+
                                               "(after 5-day smooothing), calculated over 18-year periods centered "+
                                               "around the year in which a given (smoothed using a 20-year-wide "+
                                               "gaussian kernel) GWL is first crossed in GMST")
                
                # Chunk to ~O(10MB) size chunks
                ds_tmp = ds_tmp.chunk({'lat':200,'lon':250,'gwl':-1})
            
                utility_save(ds_tmp.drop_encoding(),fns_out['mx5d_'+params_proc['bc_type']],save_kwargs={'zarr_format':3},
                             zarr_mode = 'w')
                
            # Remove placeholder file
            os.remove(placeholder_fn)
        else:
            print(', '.join([fns_out[v+'_'+params_proc['bc_type']] for v in params_param])+' exist(s), skipped!')
        
        
        # 8. Remove temporary files
        del_files = ['pctof','pctof_tiled']
        if not keep_dwscld_tseries:
            del_files.append('dwscld_'+params_proc['bc_type'])
                            
        for fn in del_files:
            if fs.exists(fns_out[fn]):
                os.system('rm -rf '+fns_out[fn])
                #fs.rm(fns_out[fn],recursive=True)
                print('Intermediary file '+fns_out[fn]+' removed!')
    #for fn in ['pctof_tiled']:
    #    if fs.exists(fns_out[fn]):
    #        fs.rm(fns_out[fn],recursive=True)
    #        print('Intermediary file '+fns_out[fn]+' removed!')

  0%|          | 0/371 [00:00<?, ?it/s]



***********************
Processing ACCESS-ESM1-5, r10i1p1f1, ssp245!
***********************
Using GWLs 0.61, 1.0, 1.5, 2.0, 2.5
/glade/derecho/scratch/schwarzwald/bcd_me/ACCESS-ESM1-5/tassumpoly_day_ACCESS-ESM1-5_ssp245_r10i1p1f1_ALLGWLS_projQDM-baseERA5-025_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/ACCESS-ESM1-5/tasbinC_day_ACCESS-ESM1-5_ssp245_r10i1p1f1_ALLGWLS_projQDM-baseERA5-025_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/ACCESS-ESM1-5/tasbinF_day_ACCESS-ESM1-5_ssp245_r10i1p1f1_ALLGWLS_projQDM-baseERA5-025_dwnsclQPLAD-target025deg.zarr exist(s), skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/ACCESS-ESM1-5/tassumpoly_day_ACCESS-ESM1-5_ssp245_r10i1p1f1_ALLGWLS_projQDM-baseGMFD_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/ACCESS-ESM1-5/tasbinC_day_ACCESS-ESM1-5_ssp245_r10i1p1f1_ALLGWLS_projQDM-baseGMFD_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/ACCESS-ESM1-5/ta

/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 2.0, 2.5, 3.0, 3.5 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in


/glade/derecho/scratch/schwarzwald/bcd_me/CNRM-ESM2-1/tassumpoly_day_CNRM-ESM2-1_ssp126_r3i1p1f2_ALLGWLS_projQDM-baseMERRA2_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/CNRM-ESM2-1/tasbinC_day_CNRM-ESM2-1_ssp126_r3i1p1f2_ALLGWLS_projQDM-baseMERRA2_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/CNRM-ESM2-1/tasbinF_day_CNRM-ESM2-1_ssp126_r3i1p1f2_ALLGWLS_projQDM-baseMERRA2_dwnsclQPLAD-target025deg.zarr exist(s), skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/CNRM-ESM2-1/tassumpoly_day_CNRM-ESM2-1_ssp126_r3i1p1f2_ALLGWLS_projQDM-baseJRA-3Q_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/CNRM-ESM2-1/tasbinC_day_CNRM-ESM2-1_ssp126_r3i1p1f2_ALLGWLS_projQDM-baseJRA-3Q_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/CNRM-ESM2-1/tasbinF_day_CNRM-ESM2-1_ssp126_r3i1p1f2_ALLGWLS_projQDM-baseJRA-3Q_dwnsclQPLAD-target025deg.zarr exist(s), skipped!


***********************
Processing CNRM-ESM2

/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 2.0, 2.5, 3.0, 3.5 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in


...

/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/bcd_me/MPI-ESM1-2-LR/tassumpoly_day_MPI-ESM1-2-LR_ssp370_r9i1p1f1_ALLGWLS_projQDM-baseERA5-025_dwnsclQPLAD-target025deg.zarr saved!
Intermediary file /glade/derecho/scratch/schwarzwald/bcd_me/MPI-ESM1-2-LR/pctoftas_day_MPI-ESM1-2-LR_ssp370_r9i1p1f1_ALLGWLS_4x.zarr removed!
/glade/derecho/scratch/schwarzwald/bcd_me/MPI-ESM1-2-LR/tassumpoly_day_MPI-ESM1-2-LR_ssp370_r9i1p1f1_ALLGWLS_projQDM-baseGMFD_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/MPI-ESM1-2-LR/tasbinC_day_MPI-ESM1-2-LR_ssp370_r9i1p1f1_ALLGWLS_projQDM-baseGMFD_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/MPI-ESM1-2-LR/tasbinF_day_MPI-ESM1-2-LR_ssp370_r9i1p1f1_ALLGWLS_projQDM-baseGMFD_dwnsclQPLAD-target025deg.zarr exist(s), skipped!
/glade/derecho/scratch/schwarzwald/bcd_me/MPI-ESM1-2-LR/tassumpoly_day_MPI-ESM1-2-LR_ssp370_r9i1p1f1_ALLGWLS_projQDM-baseMERRA2_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/MPI-ESM

/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 1.5, 2.0, 2.5 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in
/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 1.5, 2.0 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in
/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 1.5, 2.0 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in


/glade/derecho/scratch/schwarzwald/bcd_me/MRI-ESM2-0/tassumpoly_day_MRI-ESM2-0_ssp370_r3i1p1f1_ALLGWLS_projQDM-baseJRA-3Q_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/MRI-ESM2-0/tasbinC_day_MRI-ESM2-0_ssp370_r3i1p1f1_ALLGWLS_projQDM-baseJRA-3Q_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/MRI-ESM2-0/tasbinF_day_MRI-ESM2-0_ssp370_r3i1p1f1_ALLGWLS_projQDM-baseJRA-3Q_dwnsclQPLAD-target025deg.zarr exist(s), skipped!


***********************
Processing MRI-ESM2-0, r4i1p1f1, ssp245!
***********************
Using GWLs 0.61, 1.0
/glade/derecho/scratch/schwarzwald/bcd_me/MRI-ESM2-0/tassumpoly_day_MRI-ESM2-0_ssp245_r4i1p1f1_ALLGWLS_projQDM-baseERA5-025_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/MRI-ESM2-0/tasbinC_day_MRI-ESM2-0_ssp245_r4i1p1f1_ALLGWLS_projQDM-baseERA5-025_dwnsclQPLAD-target025deg.zarr, /glade/derecho/scratch/schwarzwald/bcd_me/MRI-ESM2-0/tasbinF_day_MRI-ESM2-0_ssp245_r4i1p1f1_ALLGWLS_projQDM-ba

/glade/u/home/schwarzwald/projects/bcd_me/code/funcs_aux.py:302: UserWarning: (dropping GWLs 1.5, 2.0, 2.5 since the local files do not overlap temporally with the GWL range)
  warnings.warn('(dropping GWLs '+', '.join([str(t) for t in
